# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install -U mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")


## 2. Data Overview
Review available record sets, fields, and their `@id`s.

Let's inspect the dataset and list its record sets and, for each record set, the contained fields and their `@id` values.

In [ ]:
# List record set @ids and their fields/columns
record_sets = list(dataset.record_sets)
print(f"Record sets found (by @id): {[rs.id for rs in record_sets]}")

for rs in record_sets:
    print(f"\nRecord set: {rs.name} (@id={rs.id})")
    print("Fields and columns:")
    for field in rs.fields:
        print(f"  - {field.name} (@id={field.id}, type={field.data_type})")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

In this section, we will extract all record sets using their `@id`s. For further exploration, identify which record set contains the primary survey data (usually with fields like 'age', 'phq9_score', 'gad7_score', etc.).

In [ ]:
dataframes = {}

for rs in record_sets:
    records = list(dataset.records(record_set=rs.id))
    if records:
        df = pd.DataFrame(records)
        dataframes[rs.id] = df
        print(f"Loaded {len(df)} records into DataFrame for record set '{rs.name}' (@id={rs.id}) with columns: {df.columns.tolist()}")
    else:
        print(f"No records found for record set '{rs.name}' (@id={rs.id})")

# For this dataset, assume the main record set has the first non-empty @id
main_record_set_id = next(iter(dataframes.keys())) if dataframes else None

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

**Note:** All fields and columns are referenced by their `@id`.

Below, we:
- Select a representative numeric field (e.g., PHQ-9 total score, GAD-7, or Age) by its column name/`@id` (inspect above code for canonical column names).
- Filter for high values (e.g., `phq9_score > 10`).
- Normalize that field (`z`-score).
- Group by a key attribute (e.g., `gender` or `village`).

In [ ]:
import numpy as np

# Identify a numeric field to analyze, e.g., PHQ-9 total score
# Adjust to the actual @id or column name found in the main record set
# For illustration, use 'phq9_score', 'age', or similar (verify with 'dataframes[main_record_set_id].columns')

if main_record_set_id:
    df = dataframes[main_record_set_id]
    cols = [c for c in df.columns]
    print("Columns available in record set:", cols)
    
    # Try to use a field likely to exist in a mental health survey: phq9_score, gad7_score, age
    possible_numeric_cols = ['phq9_score', 'gad7_score', 'psq_score', 'age', 'score', 'PHQ9Total']
    numeric_field = None
    for c in possible_numeric_cols:
        if c in cols:
            numeric_field = c
            break
    
    if not numeric_field:
        # Fallback: pick the first numeric column
        for c in cols:
            if pd.api.types.is_numeric_dtype(df[c]):
                numeric_field = c
                break
    
    print(f"Using numeric field for exploration: {numeric_field}")
    
    # Set an example threshold
    if numeric_field:
        threshold = np.nanpercentile(df[numeric_field], 75)  # upper quartile as a threshold
        filtered_df = df[df[numeric_field] > threshold]
        print(f"Filtered records with {numeric_field} > {threshold:.2f}:")
        display(filtered_df.head())
        
        # Normalize the numeric column
        filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
        print(f"Normalized {numeric_field} for filtered records:")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())
        
        # Group by a key attribute
        possible_group_fields = ['gender', 'village', 'level_of_education', 'marital_status']
        group_field = None
        for gf in possible_group_fields:
            if gf in filtered_df.columns:
                group_field = gf
                break
        if group_field:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean().reset_index()
            print(f"Grouped mean of {numeric_field} by {group_field}:")
            display(grouped_df)
        else:
            print("No suitable group field like 'gender', 'village', etc. found.")
    else:
        print("No suitable numeric field found for EDA.")
else:
    print("No record sets with data found.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below we plot the selected numeric field (e.g. PHQ-9 score), both as a histogram and as a boxplot grouped by a key attribute (when available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if main_record_set_id and numeric_field:
    plt.figure(figsize=(10,4))
    sns.histplot(df[numeric_field].dropna(), kde=True, bins=20)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.show()
    
    if group_field:
        plt.figure(figsize=(10,4))
        sns.boxplot(data=df, x=group_field, y=numeric_field)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.show()
else:
    print("No data or field available for visualization.")

## 6. Conclusion
This notebook demonstrates how to explore a Croissant-structured dataset using the `mlcroissant` library:
- It loaded and described the metadata and available record sets using their `@id`s as references for all operations.
- It allowed easy extraction and wrangling of survey data, providing a structure to filter, normalize, and group by key variables (e.g., survey scores, demographics).
- Initial visualizations outlined patterns in the chosen score/distribution, with the potential for more advanced analyses.

The FAIR² Mental Health Survey data enables further research into mental health patterns in Kilifi County, Kenya, serving as a template for best practices in AI-ready data sharing in African contexts.